<a href="https://colab.research.google.com/github/manoj96-alt/LoRA/blob/main/day10_full_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from google.colab import userdata
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
from openai import OpenAI
client = OpenAI(api_key=OPENAI_API_KEY)

In [5]:
import sys
sys.path.append("/content/drive/MyDrive/colab_env/lib/python3.12/site-packages")

In [6]:
from pypdf import PdfReader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain.vectorstores import FAISS
from openai import OpenAI

# Initialize LLM
client = OpenAI()

# Step 1: Load document
reader = PdfReader("/content/sample_data/Github.pdf")

text = ""
for page in reader.pages:
    text += page.extract_text()

print("\nDocument Loaded\n")

# Step 2: Chunk
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_text(text)

print(f"Chunks created: {len(chunks)}\n")

# Step 3: Embeddings
embeddings = OpenAIEmbeddings()

# Step 4: Vector DB
vector_db = FAISS.from_texts(chunks, embeddings)

print("Vector DB ready\n")

# Step 5: Chat loop
while True:

    query = input("Ask a question (or 'exit'): ")

    if query.lower() == "exit":
        print("Goodbye!")
        break

    # Step 6: Retrieve
    results = vector_db.similarity_search(query, k=3)

    context = "\n".join([doc.page_content for doc in results])

    # Step 7: Prompt
    prompt = f"""
You are an expert assistant.

Use ONLY the context below to answer.

Context:
{context}

Question:
{query}

If the answer is not in the context, say "I don't know".
"""

    # Step 8: LLM response
    response = client.responses.create(
        model="gpt-4.1-mini",
        input=prompt
    )

    print("\nAnswer:\n")
    print(response.output_text)
    print("\n" + "="*50 + "\n")

ModuleNotFoundError: No module named 'pypdf'